In [42]:
from dotenv import load_dotenv
import os
from espn_api.football import League

In [55]:
load_dotenv()

league_id = os.getenv('LEAGUE_ID')
swid = os.getenv('SWID')
espn_s2 = os.getenv('ESPN_S2')
year = 2025

In [56]:
league = League(league_id=league_id, year=year, espn_s2=espn_s2, swid=swid)


In [61]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
import json

load_dotenv()

def get_supabase_client():
    # load supabase client    
    supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
    return supabase

# supabase helper functions
def get_league_id_from_espn_league_id(supabase_client, external_league_id):
    response = supabase_client.table("leagues").select("id").eq("external_league_id", external_league_id).execute()
    return response.data[0]["id"]

def get_external_team_id_to_team_id_map(supabase_client, league_id):
    response = supabase_client.table("teams").select("id", "espn_team_id").eq("league_id", league_id).execute()
    team_map = {}
    for team in response.data:
        team_map[team["espn_team_id"]] = team["id"]
    return team_map

def get_external_player_id_to_player_id_map(supabase_client):
    response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
    player_map = {}
    for player in response.data:
        player_map[player["espn_player_id"]] = player["id"] 
    return player_map

def get_game_id_for_team_in_week(supabase_client, team_id, week, year):
    response = supabase_client.table("nfl_schedule").select("id").or_(f"home_team_id.eq.{team_id},away_team_id.eq.{team_id}").eq("week", week).eq("year", year).execute()
    
    if len(response.data) == 0:
        return None
    
    return response.data[0]["id"]

def get_team_abbrev_to_team_id_map(supabase_client):
    response = supabase_client.table("nfl_teams").select("id, team_abbrev").execute()
    team_map = {}
    for team in response.data:
        team_map[team["team_abbrev"]] = team["id"]
    return team_map

def get_player_id_to_team_id_map(supabase_client):
    response = supabase_client.table("nfl_players").select("id, team_id").execute()
    player_map = {}
    for player in response.data:
        player_map[player["id"]] = player["team_id"]
    return player_map

def get_nfl_data_player_id_to_db_player_id_map(supabase_client, nfl_data_player_id_to_espn_id_map):
    response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
    
    espn_player_id_to_id_map = {}
    nfl_data_player_id_to_id_map = {}
    for player in response.data:
        espn_player_id_to_id_map[str(player["espn_player_id"])] = player["id"]
    
    for player_id in nfl_data_player_id_to_espn_id_map:
        espn_player_id = nfl_data_player_id_to_espn_id_map[player_id]
        if espn_player_id in espn_player_id_to_id_map:
            nfl_data_player_id_to_id_map[player_id] = espn_player_id_to_id_map[espn_player_id]
        # else:
        #     print(f"Player {espn_player_id} not found in nfl_data_player_id_to_espn_id_map")
    
    return nfl_data_player_id_to_id_map
    
def get_teams_in_league(supabase_client, league_id):
    response = supabase_client.table("teams").select("id, team_name, espn_team_id").eq("league_id", league_id).execute()
    return response.data

def get_espn_player_id_to_player_id_map(supabase_client):
    response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
    player_map = {}
    for player in response.data:
        player_map[player["espn_player_id"]] = player["id"]
    return player_map

In [ ]:

supabase_client = get_supabase_client()



espn_team_id_to_team_id_map

{10: '14af6e91-3be7-4a51-9efb-a2afff769718',
 2: 'd35872ba-0aee-4bb5-8205-82b3cb35d150',
 9: 'f7fdb028-1ae4-4c7d-adec-7d4c5095a124',
 7: '77cfef79-5bd1-4767-8583-7b700e7ba8b6',
 3: 'e0ecb845-47be-446d-a5b0-29f30674d0fe',
 5: '778eec8f-f04b-4488-941d-234b41e39ea6',
 6: '713b3048-04f4-4137-8424-35caa934e14a',
 4: 'f3ad7f4a-3429-4580-be90-046edf970fba',
 8: '6f19fba8-bd44-4e29-a664-388b48e5b88a',
 1: 'd7b3bb3a-190f-4407-b43c-02f3cbb36710'}

In [67]:
def get_current_roster_for_all_teams(league, supabase_client):
    current_roster = {}
    
    db_league_id = get_league_id_from_espn_league_id(supabase_client, league.league_id)
    espn_player_id_to_player_id_map = get_espn_player_id_to_player_id_map(supabase_client)
    espn_team_id_to_team_id_map = get_external_team_id_to_team_id_map(supabase_client, db_league_id)
    
    for team in league.teams:
        current_roster[espn_team_id_to_team_id_map[team.team_id]] = []
        
        for player in team.roster:
            current_roster[espn_team_id_to_team_id_map[team.team_id]].append(espn_player_id_to_player_id_map[player.playerId])
    
    return current_roster

current_roster = get_current_roster_for_all_teams(league, supabase_client)

print(current_roster)

{'d7b3bb3a-190f-4407-b43c-02f3cbb36710': ['f92c6514-a1d2-469c-88c6-e3a5cf25e962', '0fa015c5-c14d-46d0-a228-1df96e0215dc', 'f1ae282c-2601-43aa-9aeb-c1038ab7987b', '7e289767-434f-471f-9082-2f23db948be8', '6cbcc00f-2924-4d73-9e7f-6107056575b9', '125a996c-45b2-458f-b88b-886cbcc5529f', '9e2b5ade-f88f-4c3c-b260-8799008b28c4', '8b8ca825-0903-4401-8957-bd4bcf8df197', 'd881df03-593c-4f9e-83ed-70f3a0115348', '057ac2cd-4918-4501-84f6-044d108eea6a', '5faea70c-65b9-4e8b-8f3a-df78415152a5', '9b5fa781-a12d-4501-b383-b0e68a8ac8ce', 'af32922e-d00e-4ca2-9802-19de14440d31', 'c87b9168-091f-4e25-be74-fdc479df7dc9', 'aa9f0898-8520-402f-a57e-e1dc2e0d1ccb', '7c689857-5ded-4f58-b68e-64d59003e53f'], 'd35872ba-0aee-4bb5-8205-82b3cb35d150': ['930b54f9-6314-4a7a-8798-2ff04b53d609', 'b8d5eebb-bb78-41b1-9364-8382cd976c4d', '6d2f283b-4309-4b43-bdd1-9544cce2a3de', 'b4d1f53f-515f-440e-9475-1335873e385f', 'b172eeb6-3ccb-4afb-ab6c-8d14fcc6e6d8', '3a73b8d0-2526-4431-9d6f-49f6296e8707', '6fbd1db0-6fa4-4bc0-ad05-48bb0000e28

In [53]:
for team in league.teams:
    print(team.team_name)
    print(team.roster)
    
    for player in team.roster:
        print(player.name)
        print(player.position)


TaylorMade 3
[Player(Jonathan Taylor), Player(A.J. Brown), Player(James Cook), Player(Malik Nabers), Player(Trey McBride), Player(Lamar Jackson), Player(Chase Brown), Player(Cameron Dicker), Player(Mike Evans), Player(Chargers D/ST), Player(Jaylen Warren), Player(Seahawks D/ST), Player(Tank Bigsby), Player(Nick Westbrook-Ikhine), Player(Will Levis), Player(Noah Brown), Player(Colts D/ST)]
Jonathan Taylor
RB
A.J. Brown
WR
James Cook
RB
Malik Nabers
WR
Trey McBride
TE
Lamar Jackson
QB
Chase Brown
RB
Cameron Dicker
K
Mike Evans
WR
Chargers D/ST
D/ST
Jaylen Warren
RB
Seahawks D/ST
D/ST
Tank Bigsby
RB
Nick Westbrook-Ikhine
WR
Will Levis
QB
Noah Brown
WR
Colts D/ST
D/ST
Rigged AF
[Player(Bijan Robinson), Player(Isiah Pacheco), Player(Zay Flowers), Player(Brock Bowers), Player(Steelers D/ST), Player(Garrett Wilson), Player(Jordan Love), Player(DeVonta Smith), Player(Ka'imi Fairbairn), Player(Tyrone Tracy Jr.), Player(Deebo Samuel Sr.), Player(Lions D/ST), Player(Bo Nix), Player(Jalen McMillan